<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

In [0]:
%restart_python

# Demo - Building Agent Tools on Databricks

## Overview

This demonstration focuses on how to create Unity Catalog (UC) functions using both SQL and Python for AI agent use cases and test them using AI Playground. You will combine concepts from both SQL and Python function creation, demonstrating how to build a comprehensive toolkit that leverages the strengths of each approach within a single agent workflow.

## Learning Objectives
By the end of this demo, you will be able to:
- Create and register both SQL and Python functions in Unity Catalog for Agent use cases
- Perform initial testing of your UC functions using multiple approaches
- Understand how to equip functions with proper context for AI Agent use cases
- Test your UC functions using AI Playground with multiple tools
- Identify when different tools have been used and understand how the agent utilized each tool type
- Compare the strengths of SQL vs Python tools in agent workflows

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

### A1. Install Dependencies
As part of the workspace setup, several Python libraries have been installed. To see the list of notebook-scoped libraries, please read this documentation ([AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#configure-environment-for-job-tasks) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#configure-environment-for-job-tasks) | [GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#configure-environment-for-job-tasks)). In particular, we will be leveraging the following library heavily in this demo:

1. `unitycatalog-ai[databricks]`: This package provides infrastructure and tooling for creating and managing UC functions (both SQL and Python UDFs) that can be used as tools by agents.

This demonstration uses AI Playground to test our functions, which provides a no-code interface for prototyping tool-calling agents. See the Unity Catalog Tool Integration documentation ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/unity-catalog-tool-integration) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/unity-catalog-tool-integration) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/unity-catalog-tool-integration)) for more details for advanced framework integration.

In [0]:
!pip install unitycatalog-ai[databricks]==0.3.2
%restart_python

In [0]:
%run ./Includes/Classroom-Setup-2

### A2. Inspect the Airbnb Dataset 
As a part of the classroom setup, the Airbnb dataset has been processed and stored as a Delta table within Unity Catalog. Run the next cell to query the first few rows of the dataset.

In [0]:
df = spark.read.table('sf_airbnb_listings')
display(df.limit(5))

### A3. Initialize the Databricks Function Client

Initialize the [Databricks Function Client](https://github.com/unitycatalog/unitycatalog/tree/b2d072e56661aedb84cce9be60292b2c54e12224/ai/core#databricks-managed-uc), which is a specialized interface for creating, managing, and running UC functions in Databricks. 

For building agent tools with the open source UC library, please see [this documentation](https://docs.unitycatalog.io/ai/client/#databricks-function-client). This demonstration will focus on leveraging Databricks-managed UC for building both SQL and Python agent tools.
- Supported [execution modes](https://github.com/unitycatalog/unitycatalog/blob/main/ai/core/src/unitycatalog/ai/core/utils/execution_utils.py#L25) include **Serverless** (default) and **Local**. We will use **serverless** in this notebook. 

In [0]:
from unitycatalog.ai.core.databricks import DatabricksFunctionClient

client = DatabricksFunctionClient()

## B. Define and Register UC SQL Functions

Before digging into the code, it's important to understand some terminology. 
- Built-in functions like `SUM` and `AVG` are SQL functions, but these are specifically called **system functions**. However, a **SQL function** is any reusable computation that can be called in a SQL statement, even ones defined by users.
- Any function registered via Unity Catalog, regardless of being written in SQL or Python, is considered a **user-defined function (UDF)**.

In this notebook we will use the term **SQL function** or **function** to mean a function that is or will be registered to UC, hence a SQL UDF. 

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
  <div style="display: flex; align-items: flex-start; gap: 12px;">
    <div>
      <strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
      <p style="margin: 8px 0 0 0; color: #333;">
        For more information on UDFs in Unity Catalog, please see
        this documentation (<a href="https://docs.databricks.com/aws/en/udf/unity-catalog" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/udf/unity-catalog" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/udf/unity-catalog" style="color: #1976d2; text-decoration: underline;">GCP</a>).
      </p>
    </div>
  </div>
</div>

### B1. SQL Tool: Airbnb Data Analysis

We start by creating SQL functions that will serve as our agent tools for analyzing the San Francisco Airbnb listings data. These functions include proper documentation that will help the agent understand how to use them. This tool will be created with **SQL only**. 

#### Recommendations for SQL Functions
The following SQL functions follow recommended practices:
1. **Clear parameter names and types**: Use descriptive parameter names with appropriate SQL data types
2. **Comprehensive comments**: Use `COMMENT` clauses for both the function and each parameter to provide clear descriptions
3. **Deterministic behavior**: Mark functions as `DETERMINISTIC` when they always return the same result for the same inputs
4. **Proper return type**: Explicitly specify the return data type
5. **Error handling**: Consider edge cases like NULL values in the function logic

In [0]:
%sql
CREATE OR REPLACE FUNCTION avg_neigh_price(
  neighborhood_name STRING COMMENT "The neighborhood name to filter by (e.g., 'Mission', 'Upper Market')"
)
RETURNS DOUBLE
LANGUAGE SQL
DETERMINISTIC
COMMENT 'Calculates the average listing price for a specific neighborhood in San Francisco. Returns the average price as a numeric value. Price strings are cleaned and converted to numeric values before averaging.'
RETURN 
SELECT AVG(CAST(REGEXP_REPLACE(price, '[^0-9.]', '') AS DOUBLE))
FROM sf_airbnb_listings
WHERE neighbourhood_cleansed = neighborhood_name
  AND price IS NOT NULL
  AND REGEXP_REPLACE(price, '[^0-9.]', '') != ''

### B2. Test SQL Tool Using SQL Syntax

Let's verify that our SQL function works correctly by testing it with various inputs directly in SQL. This helps ensure our function behaves as expected before integrating it with AI Playground.

In [0]:
%sql
-- Test average price function
SELECT avg_neigh_price('Mission') AS mission_avg_price

## C. Define and Register UC Python Functions

Now we'll create Python functions that complement our SQL tools by providing capabilities that would be difficult or impossible with SQL alone. Python functions enable advanced data processing, external API integration, and complex business logic. Note we can choose to wrap the Python logic in SQL syntax or use the `DatabricksFunctionClient()`. We will choose to leverage the latter since we already demonstrated how to use SQL to create a function. 

   
### C1. Best Practices for Python Agent Tools

#### Recommended Practices

1. **Explicit type hints**: Provide clear Python type hints for all arguments and the return value. This helps `create_python_function()` infer the function interface and helps agents understand expected inputs and outputs
2. **Use explicit named parameters**: Prefer clearly named, explicitly typed parameters so the tool schema is easy to understand and use
3. **Supported data types**: Ensure input and output types are compatible with Databricks and Spark SQL type systems. Refer to the supported types guidance in the Databricks documentation ([AWS](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/generative-ai/agent-framework/create-custom-tool) | [GCP](https://docs.databricks.com/gcp/en/generative-ai/agent-framework/create-custom-tool)) and [Spark SQL data types documentation](https://spark.apache.org/docs/latest/sql-ref-datatypes.html)
4. **Write clear docstrings**: Use a structured docstring format, such as [Google-style formatting](https://google.github.io/styleguide/pyguide.html#383-functions-and-methods), to describe what the function does, each argument, and the return value
    - Make descriptions precise and meaningful so agents can better determine when to use the tool
5. **Import external libraries inside the function when needed**: For functions that depend on external libraries, importing them inside the function body is the safest pattern and aligns with Databricks examples for Python UC tools and UDF-style execution

### C2. Python Tool: Extract Airbnb Listing Information

We'll create a Python function that demonstrates capabilities beyond SQL's reach - making HTTP requests to external APIs and parsing HTML content. This function will:

1. Fetch HTML content from an Airbnb posting using the listing ID
2. Extract and parse key information including description, number of reviews, and rating
3. Return formatted text that can be easily consumed by an AI agent

In [0]:
def airbnb_posting_info(id: int) -> str:
    """
    Fetches Airbnb posting information as formatted text.

    Args:
        id (int): Airbnb listing ID (e.g., 958)

    Returns:
        str: Formatted listing information (description, reviews, and rating) or error message
    """
    import requests
    import re

    api_url = f"https://www.airbnb.com/rooms/{id}"
    
    try:
        response = requests.get(api_url, timeout=10)
        
        if response.status_code == 200:
            html = response.text
            
            # Extract description
            desc = re.search(r'"metaDescription":"([^"]+)"', html)
            if desc:
                description = desc.group(1).replace('\\n', ' ')
                parts = description.split(' · ')
                description = ' · '.join(parts[2:]) if len(parts) > 2 else description
            else:
                description = "Description not found"
            
            # Extract review count and rating
            reviews = re.search(r'"reviewCount":(\d+)', html)
            rating = re.search(r'"starRating":([\d.]+)', html)
            
            reviews = reviews.group(1) if reviews else "N/A"
            rating = rating.group(1) if rating else "N/A"
            
            return f"""Description: {description}

Reviews: {reviews}
Rating: {rating} stars"""
        else:
            return f'Request failed with status code: {response.status_code}'
    
    except requests.exceptions.RequestException as e:
        return f'Request error: {str(e)}'

### C3. Test Python Function at Notebook Level

Before registering the function to Unity Catalog, it's important to test it at the notebook level to ensure it works as expected. This allows you to catch any errors early and validate the output format.

In [0]:
info = airbnb_posting_info(958)
print(info)

### C4. Register Python Tool Using `DatabricksFunctionClient()`

Now that we've validated our function works correctly, we can register it to Unity Catalog using the `DatabricksFunctionClient`. 

We use `client.create_python_function()` and pass the following parameters:

- **`func`**: The Python function object we just created (`airbnb_posting_info`)
- **`catalog`**: The catalog name, which we stored earlier as `catalog_name`
- **`schema`**: The schema name, which we stored earlier as `schema_name`
- **`replace`**: Set to `True` to overwrite the stored Python function if it already exists

In [0]:
function_info = client.create_python_function(
  func=airbnb_posting_info,
  catalog=catalog_name,
  schema=schema_name,
  replace=True
)

### C5. Test Python Tool Using `DatabricksFunctionClient()`

Let's test the Python-based function using the `execute_function()` API to ensure it works correctly when called through Unity Catalog. Note that you will receive the same response as when we performed the query for the function defined within the current Python interpreter session. 

In [0]:
result = client.execute_function(
    function_name=f"{catalog_name}.{schema_name}.airbnb_posting_info",
    parameters={"id": 958}
)

print(result.value)

## D. Discover and Test Tools via Model Context Protocol (MCP)

Now that both SQL and Python tools are registered in Unity Catalog, let's see how an agent would actually **discover** and **call** them. Databricks provides managed MCP servers that expose UC functions as tools following the open [Model Context Protocol](https://modelcontextprotocol.io/) standard.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">MCP is the <strong>recommended way</strong> to connect UC functions to agents. Rather than hard-coding function names, agents discover available tools dynamically through the MCP server.</p>
</div>
</div>
</div>

### D1. Initialize the MCP Client

`DatabricksMCPClient` connects to whatever MCP server URL you give it, including Databricks-managed MCP servers like `/api/2.0/mcp/functions/{catalog}/{schema}`.

1. Import `WorkspaceClient` from the Databricks Python SDK and create an authenticated client for the workspace.
2. Import the MCP client class `DatabricksMCPClient` and build the URL for the managed MCP server that exposes all Unity Catalog functions in the given catalog and schema.
3. Import `nest_asyncio` and call `nest_asyncio.apply()`, which patches the notebook’s running event loop so you can safely call code that uses asyncio under the hood.

In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"

4. Create a `DatabricksMCPClient` instance pointed at the managed MCP server URL. 

In [0]:
mcp_client = DatabricksMCPClient(
    server_url=mcp_url, 
    workspace_client=ws
    )

### D2. Discover Available Tools

Let's list all the tools the MCP server exposes. Notice that both our SQL function (`avg_neigh_price`) and Python function (`airbnb_posting_info`) are discoverable -- an agent would use these descriptions to decide which tool to call.

In [0]:
tools = mcp_client.list_tools()

for tool in tools:
    print(f"Tool: {tool.name}")
    print(f"  Description: {tool.description[:120]}...")
    print()

### D3. Call a Tool via MCP

We can also execute a tool through the MCP protocol, just as an agent would. Let's call `avg_neigh_price` to verify it works end-to-end through MCP.

In [0]:
tool_name = tools[1].name  # get tool name from tools list

result = mcp_client.call_tool(tool_name, {"neighborhood_name": "Mission"})

print(f"Result: {result}")

### D4. Key Takeaway

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The MCP server acts as the bridge between your UC functions and any agent framework. When you test tools in AI Playground (next section), it uses this same MCP infrastructure behind the scenes. When you build agents with LangChain or the OpenAI Agents SDK in Module 2, you'll connect to these same MCP servers.</p>
<p style="margin: 8px 0 0 0; color: #333;">This means you <strong>register once</strong> in Unity Catalog and your tools are available to any agent, any framework, with full governance.</p>
</div>
</div>
</div>

## E. Testing Combined SQL and Python Tools with AI Playground

Now that we have created and tested both SQL and Python functions, we can use them together as a comprehensive toolkit in AI Playground to create an interactive agent that can handle both data analysis and external information retrieval.

### E1. AI Playground: LLM Setup

To test your combined SQL and Python functions as agent tools in AI Playground:

1. Navigate to **Playground** from your Databricks workspace
2. Select a model with the **Tools enabled** label from the model dropdown menu at the top of the **Playground** (for example, GPT-5.1).
3. Click **Use endpoint**

### E2. Before and After Attaching Agent Tools

Before we attach any tools, let's ask a complex question that requires both data analysis and external information:
<div class="code-block-light" data-language="text">
Compare the average price in Mission with the detailed information for listing 958. Which provides better value?
</div>

The response without tools will be limited and may even outline what additional information or steps are needed to answer the question. Now, let's add our comprehensive toolkit:

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">Please make sure to open AI Playground in a new tab, select a Tools-enabled model, and add tools before running the prompt. It's recommended that you try your tools on the latest released models (e.g. GPT, Claude, etc.).</p>
</div>
</div>
</div>

1. Click **Tools > + Add tool**
2. Under **MCP Servers**, click on **Browse** under **Unity Catalog Function**. 
    - In the search box, search for and click on the function `avg_neigh_price`
3. Click **Confirm** at the bottom right
5. Repeat the steps above to add `airbnb_posting_info`
6. Click **Save**
7. Validate that all tools are equipped; you should see **Tools (2)** in the **Tools** dropdown menu

Now that we have our comprehensive toolkit attached, let's examine how the agent intelligently selects and combines different tool types. Ask the question again:

<div class="code-block-light" data-language="text">
Compare the average price in Mission with the detailed information for listing 958. Which provides better value?
</div>

You can now observe how the agent:
1. **Uses the SQL tool** to get average pricing data from the database
2. **Uses the Python tool** to fetch external information about the specific listing
3. **Combines the results** to provide a comprehensive analysis

The agent reasoning will show something like:
- First tool call: SQL function to get Mission average price
- Second tool call: Python function to get listing 958 details
- Analysis combining both data sources

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

## Conclusion

You've now learned how to create a comprehensive AI agent toolkit by combining both SQL and Python functions in Unity Catalog. Throughout this demonstration, you've gained hands-on experience with:

- **Building both SQL and Python UC functions** for AI agents using multiple registration approaches
- **Understanding the strengths of each approach** - SQL for data analysis and Python for external integrations and complex logic
- **Testing functions individually and collectively** through multiple methods including direct execution, `DatabricksFunctionClient()`, and AI Playground
- **Creating comprehensive agent toolkits** that can handle diverse user queries requiring both data analysis and external information
- **Monitoring multi-tool agent behavior** to understand how agents intelligently select and combine different tool types

By combining Unity Catalog's governance framework with both SQL's analytical capabilities and Python's computational flexibility, you're now equipped to build secure, scalable AI agent solutions that can intelligently interact with both your internal data and external systems. This comprehensive approach enables you to create production-ready agent tools that leverage the best of both worlds while maintaining enterprise governance and security standards.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>